In [2]:
%matplotlib inline

import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path().resolve().parent   
sys.path.insert(0, str(ROOT / "src"))

from modem.pipeline import run
from modem.blocks import (
    TextSource, RawBitSource, HexSource,
    BytesToBits, BitsToBytes, MessageSink,
    BPSKModulator, BPSKDemodulator, DBPSKDemodulator,
    QPSKModulator, QPSKDemodulator, DQPSKModulator, DQPSKDemodulator,
    BFSKModulator, BFSKDemodulator, DBFSKDemodulator,
    DummyModulator, DummyDemodulator,
    IQModulator, IQDemodulator,
    ZeroOrderHold, IntegrateAndDump,
    AWGNChannel, RayleighFlatFadingChannel, MultipathChannel, NoChannel, DopplerChannel, FrequencyCorrector,
    PrependBit, DifferentialEncoder,
    HammingCoder, HammingDecoder, DummyCoder, DummyDecoder,
    LMSLinearEqualizer,GFSKModulator,
    PreambleCorrelator, BLE_GFSK_Modulator, BLE_GFSK_Demodulator,
)

# Вспомогательные блоки (можно вынести в отдельный модуль, но пока так)
from modem import Block
class PrependArrayBlock(Block):
    def __init__(self, prepend, name="prepend"):
        super().__init__(name)
        self.prepend = np.asarray(prepend)
    def process(self, x, **kwargs):
        return np.concatenate([self.prepend, x])

class RemovePreambleBlock(Block):
    def __init__(self, n, name="remove"):
        super().__init__(name)
        self.n = n
    def process(self, x, **kwargs):
        return x[self.n:]

print("Импорты успешны")

Импорты успешны


In [3]:
import ipywidgets as widgets
from IPython.display import display

source_widget = widgets.Dropdown(
    options=[('Текст', 'text'), ('Биты', 'bits'), ('Hex', 'hex')],
    value='text',
    description='Источник:')

message_widget = widgets.Text(value='Hello, world!', description='Текст/биты:')

coding_widget = widgets.Dropdown(
    options=[('Хэмминг (7,4)', 'hamming'), ('Без кодирования', 'none')],
    value='hamming',
    description='Код:')

mod_widget = widgets.Dropdown(
    options=[('BPSK', 'bpsk'), ('DBPSK', 'dbpsk'),
             ('QPSK', 'qpsk'), ('DQPSK', 'dqpsk'),
             ('BFSK', 'bfsk'), ('DBFSK', 'dbfsk'),    ('GFSK', 'gfsk'),
             ('Без модуляции', 'none')],
    value='bpsk',
    description='Модуляция:')

channel_widget = widgets.Dropdown(
    options=[('AWGN', 'awgn'), ('Rayleigh + AWGN', 'rayleigh'),
             ('Многолучевой (с эквалайзером)', 'multipath'),('Без шума','none')],
    value='awgn',
    description='Канал:')

snr_widget = widgets.FloatSlider(value=20, min=-60, max=60, step=0.5, description='SNR (dB):')
sps_widget = widgets.IntSlider(value=20, min=1, max=200, step=1, description='Сэмплов/символ:')
fs_widget = widgets.FloatText(value=1000.0, description='Част. дискр.:')
fc_widget = widgets.FloatText(value=50.0, description='Несущая (Гц):')
use_carrier_widget = widgets.Checkbox(value=False, description='Перенос на несущую')

deviation_widget = widgets.FloatText(value=25.0, description='FSK девиация:')

block_size_widget = widgets.IntText(value=200, description='Размер блока:')
multipath_widget = widgets.Text(value='4,0.6', description='Лучи (задержка,коэф):')

freq_offset_widget = widgets.FloatSlider(
    value=0, min=0, max=5000, step=1,
    description='Допплер (Гц):',
    continuous_update=False
)
compensate_widget = widgets.Checkbox(value=False, description='Компенсировать допплер')

run_button = widgets.Button(description='Запустить моделирование', button_style='success')
block_view_widget = widgets.Dropdown(
    options=[
        'coder',
        'mod',
        'awgn',
        'demod',
        'decoder'
    ],
    value='mod',
    description='Блок:'
)

spectrum_widget = widgets.Dropdown(
    options=[
        ('Амплитудный', 'amplitude'),
        ('Фазовый', 'phase')
    ],
    value='amplitude',
    description='Спектр:'
)
# Группируем виджеты
out = widgets.Output()
ui = widgets.VBox([
    widgets.HBox([source_widget, message_widget]),
    widgets.HBox([coding_widget, mod_widget, channel_widget]),
    snr_widget, sps_widget,
    widgets.HBox([fs_widget, fc_widget, use_carrier_widget]),
    deviation_widget,
    block_size_widget, multipath_widget,
    freq_offset_widget, compensate_widget, 
    widgets.HBox([
        block_view_widget,
        spectrum_widget
    ]),
    run_button,
    out  # сюда будут падать результаты
])
display(ui)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from modem.blocks import BLE_GFSK_Modulator, BLE_GFSK_Demodulator

# ============================================================
# 1. Вспомогательные функции
# ============================================================

# ---------- НОВОЕ: bt_swap_bits и девайтенинг ----------
def bt_swap_bits(byte: int) -> int:
    """Переворачивает порядок битов в байте (как в BLE)."""
    result = 0
    for i in range(8):
        result |= ((byte >> i) & 1) << (7 - i)
    return result

def ll_packet_data_dewhitening(data: bytes, channel: int) -> bytes:
    """Выполняет вайтнинг/девайтнинг данных BLE на основе переданного Си-алгоритма."""
    length = len(data)
    lfsr = (bt_swap_bits(channel) | 2) & 0xFF
    output = bytearray(data)

    for i in range(length):
        d = bt_swap_bits(output[i])
        j = 128
        while j >= 1:
            if lfsr & 0x80:
                lfsr ^= 0x11
                d ^= j
            lfsr = (lfsr << 1) & 0xFF
            j >>= 1
        output[i] = bt_swap_bits(d)

    return bytes(output)
# ----------------------------------------------------------

def crc24_ble(data: bytes) -> int:
    """24‑битный CRC для BLE."""
    poly = 0x100065b
    crc = 0x555555
    for byte in data:
        crc ^= (byte << 16)
        for _ in range(8):
            crc <<= 1
            if crc & 0x1000000:
                crc ^= poly
    return crc & 0xFFFFFF

# ============================================================
# 2. Основная функция анализа
# ============================================================

def run_ble_analysis(filename, offset, whitening_channel):
    fs = 8e6
    sps = 8
    dev = 250e3

    # --- BLE preamble ---
    sync_byte = 0xAA
    access_addr = 0x8E89BED6
    sync_bits = np.unpackbits(np.array([sync_byte], dtype=np.uint8), bitorder="little")
    addr_bytes = access_addr.to_bytes(4, byteorder="little")
    addr_bits = np.unpackbits(np.frombuffer(addr_bytes, dtype=np.uint8), bitorder="little")
    preamble_bits = np.concatenate([sync_bits, addr_bits])

    mod = BLE_GFSK_Modulator(fs=fs, deviation_hz=dev, samples_per_symbol=sps)
    ideal_pre = mod(preamble_bits)

    # --- Загрузка IQ ---
    real_sig = np.fromfile(filename, dtype=np.complex64)
    print(f"Длина сигнала: {len(real_sig)} отсчётов")

    # --- Корреляция сигнала ---
    corr_sig = np.correlate(real_sig, ideal_pre, mode='same')
    peak_sample = np.argmax(np.abs(corr_sig))
    print(f"Пик корреляции (отсчёты): {peak_sample}")

    start_sample = peak_sample - len(ideal_pre)//2
    if start_sample < 0:
        start_sample = 0

    # --- Демодуляция короткого участка ---
    total_bits = len(preamble_bits) + 2000
    segment_len = total_bits * sps
    segment = real_sig[start_sample : start_sample + segment_len]

    demod = BLE_GFSK_Demodulator(sps=sps, fixed_offset=offset)
    rx_bits = demod(segment)

    print(f"Демодулировано бит: {len(rx_bits)}")


    # --- Поиск преамбулы в битах ---
    rx_pm = 2 * rx_bits.astype(np.int16) - 1
    pre_pm = 2 * preamble_bits.astype(np.int16) - 1
    corr_bits = np.correlate(rx_pm, pre_pm, mode='valid')
    peak_bit = np.argmax(np.abs(corr_bits))
    peak_val = np.max(np.abs(corr_bits))

 
    print(f"Преамбула в битах, пик: {peak_bit}, значение: {peak_val}")
    print("\n=== Проверка синхронизации ===")
    print("start_sample =", start_sample)
    print("peak_bit =", peak_bit)
    print("peak_bit*sps =", peak_bit * sps)

    rx_pre = rx_bits[peak_bit : peak_bit + len(preamble_bits)]
    errors = np.sum(rx_pre != preamble_bits)
    print(f"Ошибок в преамбуле: {errors}")

    # --- Вывод декодированной преамбулы ---
    print("\n--- Декодированная преамбула ---")
    if errors > 0:
        diff_pos = np.where(rx_pre != preamble_bits)[0]
        print(f"Ошибочные позиции: {diff_pos}")
    rx_pre_bytes = np.packbits(rx_pre, bitorder='little')
    et_pre_bytes = np.packbits(preamble_bits, bitorder='little')
    print("Hex (декодировано):", ' '.join(f'{b:02X}' for b in rx_pre_bytes))
    print("Hex (эталон):       ", ' '.join(f'{b:02X}' for b in et_pre_bytes))

    next_bits = rx_bits[
        peak_bit + len(preamble_bits):
        peak_bit + len(preamble_bits) + 160
    ]


    print("\nПервые 10 байт после Access Address (до девайтенинга):")
    tmp_bits = next_bits.copy()
    if len(tmp_bits) % 8:
        pad = 8 - len(tmp_bits) % 8
        tmp_bits = np.concatenate([tmp_bits, np.zeros(pad, dtype=np.uint8)])
    tmp_bytes = np.packbits(tmp_bits, bitorder="little")
    print(' '.join(f'{b:02X}' for b in tmp_bytes))

    # ---------- НОВОЕ: применяем девайтенинг ----------
    dewhitened_bytes = ll_packet_data_dewhitening(bytes(tmp_bytes), whitening_channel)
    print("\nПосле девайтенинга:")
    print(' '.join(f'{b:02X}' for b in dewhitened_bytes))
    # ----------------------------------------------------
    
    # --- Графики ---
    fig, axes = plt.subplots(3, 1, figsize=(14, 10))

    axes[0].plot(real_sig[:4000].real, label='I')
    axes[0].plot(real_sig[:4000].imag, label='Q')
    if start_sample < 4000:
        axes[0].axvline(start_sample, color='r', linestyle='--', label='начало преамбулы')
    axes[0].set_title("IQ сигнал")
    axes[0].grid(True); axes[0].legend()

    axes[1].plot(np.abs(corr_sig))
    axes[1].axvline(peak_sample, color='r', linestyle='--')
    axes[1].set_title(f"Корреляция сигнала (offset={offset})")
    axes[1].grid(True)

    pre_segment = real_sig[start_sample : start_sample + len(ideal_pre)]
    t = np.arange(len(pre_segment))
    axes[2].plot(t, pre_segment.real, label='I real')
    axes[2].plot(t, pre_segment.imag, label='Q real')
    axes[2].plot(t, ideal_pre.real, '--', label='I ideal')
    axes[2].plot(t, ideal_pre.imag, '--', label='Q ideal')
    axes[2].set_title("Найденная преамбула")
    axes[2].grid(True); axes[2].legend()

    plt.tight_layout()
    plt.show()

# ============================================================
# Виджеты
# ============================================================
ble_file_widget = widgets.Text(
    value=r"C:/Users/Professional/Desktop/BTLE_packet_2_fs_8e6.dat",
    description="IQ файл:",
    layout=widgets.Layout(width="900px")
)
ble_offset_widget = widgets.IntSlider(value=0, min=0, max=7, step=1, description="Offset:")
ble_channel_widget = widgets.IntSlider(value=38, min=0, max=39, step=1, description="Whitening:")
ble_button = widgets.Button(description="BLE анализ", button_style="success")
ble_output = widgets.Output()

def on_ble_click(b):
    with ble_output:
        ble_output.clear_output()
        run_ble_analysis(ble_file_widget.value, ble_offset_widget.value, ble_channel_widget.value)

ble_button.on_click(on_ble_click)

display(widgets.VBox([ble_file_widget, ble_offset_widget, ble_channel_widget, ble_button, ble_output]))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from modem.blocks import BLE_GFSK_Modulator, BLE_GFSK_Demodulator

# ============================================================
# Общие вспомогательные функции
# ============================================================
def bt_swap_bits(byte: int) -> int:
    """Переворачивает порядок битов в байте (как в BLE)."""
    result = 0
    for i in range(8):
        result |= ((byte >> i) & 1) << (7 - i)
    return result

def ll_packet_data_dewhitening(data: bytes, channel: int) -> bytes:
    """
    Вайтенинг / девайтенинг данных BLE.
    Операция симметрична – используется и для whitening, и для dewhitening.
    """
    length = len(data)
    lfsr = (bt_swap_bits(channel) | 2) & 0xFF
    output = bytearray(data)

    for i in range(length):
        d = bt_swap_bits(output[i])
        j = 128
        while j >= 1:
            if lfsr & 0x80:
                lfsr ^= 0x11
                d ^= j
            lfsr = (lfsr << 1) & 0xFF
            j >>= 1
        output[i] = bt_swap_bits(d)

    return bytes(output)

def crc24_ble(data: bytes) -> int:
    """24‑битный CRC для BLE."""
    poly = 0x100065b
    crc = 0x555555
    for byte in data:
        crc ^= (byte << 16)
        for _ in range(8):
            crc <<= 1
            if crc & 0x1000000:
                crc ^= poly
    return crc & 0xFFFFFF

# ============================================================
# 1. Функция АНАЛИЗА (демодуляция существующего файла)
# ============================================================
def run_ble_analysis(filename, offset, whitening_channel):
    fs = 8e6
    sps = 8
    dev = 250e3

    # --- BLE preamble ---
    sync_byte = 0xAA
    access_addr = 0x8E89BED6
    sync_bits = np.unpackbits(np.array([sync_byte], dtype=np.uint8), bitorder="little")
    addr_bytes = access_addr.to_bytes(4, byteorder="little")
    addr_bits = np.unpackbits(np.frombuffer(addr_bytes, dtype=np.uint8), bitorder="little")
    preamble_bits = np.concatenate([sync_bits, addr_bits])

    mod = BLE_GFSK_Modulator(fs=fs, deviation_hz=dev, samples_per_symbol=sps)
    ideal_pre = mod(preamble_bits)

    # --- Загрузка IQ ---
    real_sig = np.fromfile(filename, dtype=np.complex64)
    print(f"Длина сигнала: {len(real_sig)} отсчётов")

    # --- Корреляция сигнала ---
    corr_sig = np.correlate(real_sig, ideal_pre, mode='same')
    peak_sample = np.argmax(np.abs(corr_sig))
    print(f"Пик корреляции (отсчёты): {peak_sample}")

    start_sample = peak_sample - len(ideal_pre)//2
    if start_sample < 0:
        start_sample = 0

    # --- Демодуляция короткого участка ---
    total_bits = len(preamble_bits) + 2000
    segment_len = total_bits * sps
    segment = real_sig[start_sample : start_sample + segment_len]

    demod = BLE_GFSK_Demodulator(sps=sps, fixed_offset=offset)
    rx_bits = demod(segment)

    print(f"Демодулировано бит: {len(rx_bits)}")

    # --- Поиск преамбулы в битах ---
    rx_pm = 2 * rx_bits.astype(np.int16) - 1
    pre_pm = 2 * preamble_bits.astype(np.int16) - 1
    corr_bits = np.correlate(rx_pm, pre_pm, mode='valid')
    peak_bit = np.argmax(np.abs(corr_bits))
    peak_val = np.max(np.abs(corr_bits))

    print(f"Преамбула в битах, пик: {peak_bit}, значение: {peak_val}")
    print("\n=== Проверка синхронизации ===")
    print("start_sample =", start_sample)
    print("peak_bit =", peak_bit)
    print("peak_bit*sps =", peak_bit * sps)

    rx_pre = rx_bits[peak_bit : peak_bit + len(preamble_bits)]
    errors = np.sum(rx_pre != preamble_bits)
    print(f"Ошибок в преамбуле: {errors}")

    # --- Декодированная преамбула ---
    print("\n--- Декодированная преамбула ---")
    if errors > 0:
        diff_pos = np.where(rx_pre != preamble_bits)[0]
        print(f"Ошибочные позиции: {diff_pos}")
    rx_pre_bytes = np.packbits(rx_pre, bitorder='little')
    et_pre_bytes = np.packbits(preamble_bits, bitorder='little')
    print("Hex (декодировано):", ' '.join(f'{b:02X}' for b in rx_pre_bytes))
    print("Hex (эталон):       ", ' '.join(f'{b:02X}' for b in et_pre_bytes))

    next_bits = rx_bits[
        peak_bit + len(preamble_bits):
        peak_bit + len(preamble_bits) + 160
    ]

    print("\nПервые 10 байт после Access Address (до девайтенинга):")
    tmp_bits = next_bits.copy()
    if len(tmp_bits) % 8:
        pad = 8 - len(tmp_bits) % 8
        tmp_bits = np.concatenate([tmp_bits, np.zeros(pad, dtype=np.uint8)])
    tmp_bytes = np.packbits(tmp_bits, bitorder="little")
    print(' '.join(f'{b:02X}' for b in tmp_bytes))

    # Девайтенинг
    dewhitened_bytes = ll_packet_data_dewhitening(bytes(tmp_bytes), whitening_channel)
    print("\nПосле девайтенинга:")
    print(' '.join(f'{b:02X}' for b in dewhitened_bytes))

    # --- Графики ---
    fig, axes = plt.subplots(3, 1, figsize=(14, 10))

    axes[0].plot(real_sig[:4000].real, label='I')
    axes[0].plot(real_sig[:4000].imag, label='Q')
    if start_sample < 4000:
        axes[0].axvline(start_sample, color='r', linestyle='--', label='начало преамбулы')
    axes[0].set_title("IQ сигнал")
    axes[0].grid(True); axes[0].legend()

    axes[1].plot(np.abs(corr_sig))
    axes[1].axvline(peak_sample, color='r', linestyle='--')
    axes[1].set_title(f"Корреляция сигнала (offset={offset})")
    axes[1].grid(True)

    pre_segment = real_sig[start_sample : start_sample + len(ideal_pre)]
    t = np.arange(len(pre_segment))
    axes[2].plot(t, pre_segment.real, label='I real')
    axes[2].plot(t, pre_segment.imag, label='Q real')
    axes[2].plot(t, ideal_pre.real, '--', label='I ideal')
    axes[2].plot(t, ideal_pre.imag, '--', label='Q ideal')
    axes[2].set_title("Найденная преамбула")
    axes[2].grid(True); axes[2].legend()

    plt.tight_layout()
    plt.show()

# ============================================================
# 2. Функция ГЕНЕРАЦИИ синтетического пакета
# ============================================================
def synthesize_ble_packet(
    channel: int = 38,
    access_addr: int = 0x8E89BED6,
    pdu_payload: bytes = b'\x11\x22\x33\x44\x55\x66',
    pdu_type: int = 0,
    tx_add: int = 0,
    rx_add: int = 0,
    offset_hz: float = 15000.0,
    snr_db: float = 35.0,
    fs: float = 8e6,
    sps: int = 8,
    deviation_hz: float = 250e3
) -> np.ndarray:
    payload_len = len(pdu_payload)
    if not (0 <= payload_len <= 37):
        raise ValueError("Длина payload должна быть 0..37 байт")

    # Заголовок PDU
    pdu_type_bits = [(pdu_type >> i) & 1 for i in range(4)]
    rfu_bit = [0]
    ch_sel_bit = [0]
    tx_add_bit = [tx_add]
    rx_add_bit = [rx_add]
    length_bits = [(payload_len >> i) & 1 for i in range(6)]

    header_bits = np.array(
        pdu_type_bits + rfu_bit + ch_sel_bit + tx_add_bit + rx_add_bit + length_bits,
        dtype=np.uint8
    )
    header_bytes = np.packbits(header_bits, bitorder='little').tobytes()

    # Сборка PDU и CRC
    pdu = header_bytes + pdu_payload
    crc_val = crc24_ble(pdu)
    crc_bytes = crc_val.to_bytes(3, byteorder='little')

    # Вайтенинг
    data_to_whiten = pdu + crc_bytes
    whitened = ll_packet_data_dewhitening(data_to_whiten, channel)

    # Полный пакет байтов
    preamble = b'\xAA'
    addr_bytes = access_addr.to_bytes(4, byteorder='little')
    packet_bytes = preamble + addr_bytes + whitened

    # Биты LSB first
    bits = np.unpackbits(np.frombuffer(packet_bytes, dtype=np.uint8), bitorder='little')

    # Модуляция
    mod = BLE_GFSK_Modulator(fs=fs, deviation_hz=deviation_hz, samples_per_symbol=sps)
    iq_signal = mod(bits)

    # Частотный сдвиг
    if offset_hz != 0.0:
        t = np.arange(len(iq_signal)) / fs
        iq_signal *= np.exp(1j * 2.0 * np.pi * offset_hz * t)

    # Шум
    if snr_db is not None:
        signal_power = np.mean(np.abs(iq_signal)**2)
        snr_linear = 10.0 ** (snr_db / 10.0)
        noise_power = signal_power / snr_linear
        noise = np.sqrt(noise_power / 2) * (
            np.random.randn(len(iq_signal)) + 1j * np.random.randn(len(iq_signal))
        )
        iq_signal += noise

    return iq_signal.astype(np.complex64)

# ============================================================
# 3. Построение общего интерфейса
# ============================================================
mode_selector = widgets.RadioButtons(
    options=['Анализ файла', 'Генерация сигнала'],
    value='Анализ файла',
    description='Режим:'
)

# Виджеты для анализа
analysis_file = widgets.Text(
    value=r"C:/Users/Professional/Desktop/BTLE_packet_2_fs_8e6.dat",
    description="IQ файл:",
    layout=widgets.Layout(width="800px")
)
analysis_offset = widgets.IntSlider(value=0, min=0, max=7, step=1, description="Offset:")
analysis_channel = widgets.IntSlider(value=38, min=0, max=39, step=1, description="Whitening channel:")
analysis_button = widgets.Button(description="Запустить анализ", button_style="success")

# Виджеты для генерации
gen_channel = widgets.IntSlider(value=38, min=0, max=39, step=1, description='Channel:')
gen_access_addr = widgets.Text(value='8E89BED6', description='Access Addr (hex):')
gen_payload = widgets.Text(value='112233445566', description='Payload (hex):')
gen_offset = widgets.FloatSlider(value=15000, min=0, max=100000, step=100,
                                 description='Offset (Hz):', readout_format='.0f')
gen_snr = widgets.FloatSlider(value=35, min=1, max=60, step=1, description='SNR (dB):')
gen_output_file = widgets.Text(value='synthetic_ble_packet.dat', description='Output file:')
gen_button = widgets.Button(description="Сгенерировать и сохранить", button_style="success")

# Контейнеры для переключения
analysis_box = widgets.VBox([analysis_file, analysis_offset, analysis_channel, analysis_button])
gen_box = widgets.VBox([gen_channel, gen_access_addr, gen_payload, gen_offset, gen_snr,
                        gen_output_file, gen_button])

ui_box = widgets.VBox([mode_selector, analysis_box])  # начальное отображение — анализ
output_area = widgets.Output()

# Обработчики нажатий
def on_analysis_clicked(b):
    with output_area:
        clear_output()
        run_ble_analysis(analysis_file.value, analysis_offset.value, analysis_channel.value)

def on_gen_clicked(b):
    with output_area:
        clear_output()
        try:
            access_addr = int(gen_access_addr.value, 16)
            payload_hex = gen_payload.value.replace(' ', '')
            payload = bytes.fromhex(payload_hex)
            iq = synthesize_ble_packet(
                channel=gen_channel.value,
                access_addr=access_addr,
                pdu_payload=payload,
                offset_hz=gen_offset.value,
                snr_db=gen_snr.value
            )
            iq.tofile(gen_output_file.value)
            print(f"Файл '{gen_output_file.value}' сохранён, длина {len(iq)} отсчётов.")
            # Визуализация первых 5000 отсчётов
            fig, ax = plt.subplots(1, 1, figsize=(10, 4))
            ax.plot(iq[:5000].real, label='I')
            ax.plot(iq[:5000].imag, label='Q')
            ax.set_title("Сгенерированный IQ (первые 5000 отсчётов)")
            ax.grid(True)
            ax.legend()
            plt.show()
        except Exception as e:
            print(f"Ошибка: {e}")

analysis_button.on_click(on_analysis_clicked)
gen_button.on_click(on_gen_clicked)

# Переключение контейнеров при смене режима
def on_mode_change(change):
    if change['new'] == 'Анализ файла':
        ui_box.children = [mode_selector, analysis_box]
    else:
        ui_box.children = [mode_selector, gen_box]

mode_selector.observe(on_mode_change, names='value')

# Отображение всего интерфейса
display(ui_box)
display(output_area)

Output()

In [12]:
import numpy as np
from modem.blocks import BLE_GFSK_Modulator

# ============================================================
# Вспомогательные функции (те же, что в исходном коде)
# ============================================================

def bt_swap_bits(byte: int) -> int:
    """Переворачивает порядок битов в байте (как в BLE)."""
    result = 0
    for i in range(8):
        result |= ((byte >> i) & 1) << (7 - i)
    return result

def ll_packet_data_whitening(data: bytes, channel: int) -> bytes:
    """
    Вайтенинг/девайтенинг данных BLE.
    Операция симметрична, поэтому используется для вайтенинга.
    """
    length = len(data)
    lfsr = (bt_swap_bits(channel) | 2) & 0xFF
    output = bytearray(data)

    for i in range(length):
        d = bt_swap_bits(output[i])
        j = 128
        while j >= 1:
            if lfsr & 0x80:
                lfsr ^= 0x11
                d ^= j
            lfsr = (lfsr << 1) & 0xFF
            j >>= 1
        output[i] = bt_swap_bits(d)

    return bytes(output)

def crc24_ble(data: bytes) -> int:
    """24‑битный CRC для BLE."""
    poly = 0x100065b
    crc = 0x555555
    for byte in data:
        crc ^= (byte << 16)
        for _ in range(8):
            crc <<= 1
            if crc & 0x1000000:
                crc ^= poly
    return crc & 0xFFFFFF

# ============================================================
# Функция синтеза одного BLE-пакета
# ============================================================

def synthesize_ble_packet(
    channel: int = 38,
    access_addr: int = 0x8E89BED6,
    pdu_payload: bytes = b'\x11\x22\x33\x44\x55\x66',  # 6-байтовый AdvA
    pdu_type: int = 0,       # 0 = ADV_IND
    tx_add: int = 0,
    rx_add: int = 0,
    offset_hz: float = 15000.0,   # частотный сдвиг в герцах
    snr_db: float = 35.0,         # SNR в дБ, если None – без шума
    fs: float = 8e6,
    sps: int = 8,
    deviation_hz: float = 250e3
) -> np.ndarray:
    """
    Генерирует комплексный IQ-сигнал BLE-пакета (формат complex64).

    Параметры
    ---------
    channel : номер канала (0..39)
    access_addr : 32-битный Access Address
    pdu_payload : байты полезной нагрузки PDU (без заголовка и CRC)
    pdu_type : тип PDU (4 бита), по умолчанию 0 (ADV_IND)
    tx_add, rx_add : биты TxAdd/RxAdd заголовка
    offset_hz : частотный сдвиг несущей (Гц)
    snr_db : SNR (дБ), если None, шум не добавляется
    fs, sps, deviation_hz : параметры модуляции
    """
    # --- 1. Формируем заголовок PDU (2 байта) ---
    payload_len = len(pdu_payload)  # до 37 байт (в рекламном канале обычно 6..31)
    if not (0 <= payload_len <= 37):
        raise ValueError("Длина payload должна быть 0..37 байт")

    # Биты заголовка в порядке передачи (LSB first)
    pdu_type_bits = [(pdu_type >> i) & 1 for i in range(4)]    # 4 бита типа
    rfu_bit = [0]                                               # RFU (зарезервирован)
    ch_sel_bit = [0]                                            # ChSel (не используется в LE 1M)
    tx_add_bit = [tx_add]
    rx_add_bit = [rx_add]
    length_bits = [(payload_len >> i) & 1 for i in range(6)]   # длина (6 бит)

    header_bits = np.array(
        pdu_type_bits + rfu_bit + ch_sel_bit + tx_add_bit + rx_add_bit + length_bits,
        dtype=np.uint8
    )
    # Упаковываем в 2 байта (LSB first)
    header_bytes = np.packbits(header_bits, bitorder='little').tobytes()

    # --- 2. Собираем PDU и вычисляем CRC ---
    pdu = header_bytes + pdu_payload
    crc_val = crc24_ble(pdu)
    crc_bytes = crc_val.to_bytes(3, byteorder='little')  # младший байт первым

    # --- 3. Применяем вайтенинг к PDU+CRC ---
    data_to_whiten = pdu + crc_bytes
    whitened = ll_packet_data_whitening(data_to_whiten, channel)

    # --- 4. Формируем полный пакет байтов ---
    preamble = b'\xAA'  # 1 байт преамбулы (10101010, LSB first)
    addr_bytes = access_addr.to_bytes(4, byteorder='little')  # LSB first
    packet_bytes = preamble + addr_bytes + whitened

    # --- 5. Преобразуем байты в биты (LSB first) ---
    bits = np.unpackbits(np.frombuffer(packet_bytes, dtype=np.uint8), bitorder='little')

    # --- 6. Модуляция GFSK ---
    mod = BLE_GFSK_Modulator(fs=fs, deviation_hz=deviation_hz, samples_per_symbol=sps)
    iq_signal = mod(bits)

    # --- 7. Частотный сдвиг ---
    if offset_hz != 0.0:
        t = np.arange(len(iq_signal)) / fs
        iq_signal *= np.exp(1j * 2.0 * np.pi * offset_hz * t)

    # --- 8. Добавление шума ---
    if snr_db is not None:
        # Средняя мощность сигнала
        signal_power = np.mean(np.abs(iq_signal)**2)
        # Мощность шума для заданного SNR
        snr_linear = 10.0 ** (snr_db / 10.0)
        noise_power = signal_power / snr_linear
        # Комплексный гауссовский шум
        noise = np.sqrt(noise_power / 2) * (
            np.random.randn(len(iq_signal)) + 1j * np.random.randn(len(iq_signal))
        )
        iq_signal += noise

    return iq_signal.astype(np.complex64)

# ============================================================
# Виджеты для интерактивной генерации (можно использовать отдельно)
# ============================================================
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

channel_w = widgets.IntSlider(value=38, min=0, max=39, step=1, description='Channel:')
access_addr_w = widgets.Text(value='8E89BED6', description='Access Addr (hex):')
payload_w = widgets.Text(value='112233445566', description='Payload (hex):')
offset_w = widgets.FloatSlider(value=15000, min=0, max=100000, step=100, description='Offset (Hz):',
                               readout_format='.0f')
snr_w = widgets.FloatSlider(value=35, min=0, max=60, step=1, description='SNR (dB):')
output_dir_w = widgets.Text(value='synthetic_ble_packet.dat', description='Output file:')
gen_button = widgets.Button(description='Сгенерировать и сохранить', button_style='success')
output_plot = widgets.Output()

def on_generate_clicked(b):
    with output_plot:
        clear_output()
        try:
            access_addr = int(access_addr_w.value, 16)
            payload_hex = payload_w.value.replace(' ', '')
            payload = bytes.fromhex(payload_hex)
            iq = synthesize_ble_packet(
                channel=channel_w.value,
                access_addr=access_addr,
                pdu_payload=payload,
                offset_hz=offset_w.value,
                snr_db=snr_w.value
            )
            iq.tofile(output_dir_w.value)
            print(f"Файл '{output_dir_w.value}' сохранён, длина {len(iq)} отсчётов.")

            # Быстрая визуализация первых 5000 отсчётов
            fig, ax = plt.subplots(1, 1, figsize=(10, 4))
            ax.plot(iq[:5000].real, label='I')
            ax.plot(iq[:5000].imag, label='Q')
            ax.set_title("Сгенерированный IQ (первые 5000 отсчётов)")
            ax.grid(True)
            ax.legend()
            plt.show()
        except Exception as e:
            print(f"Ошибка: {e}")

gen_button.on_click(on_generate_clicked)

widgets.VBox([channel_w, access_addr_w, payload_w, offset_w, snr_w, output_dir_w, gen_button, output_plot])

In [ ]:
def build_blocks_from_widgets():
    # Источник
    if source_widget.value == 'text':
        src = TextSource(name="source", message=message_widget.value)
    elif source_widget.value == 'bits':
        src = RawBitSource(name="source", bitstring=message_widget.value)
    else:
        src = HexSource(name="source", hexstring=message_widget.value)

    # Кодер/декодер
    if coding_widget.value == 'hamming':
        coder = HammingCoder(name="coder")
        decoder = HammingDecoder(name="decoder")
    else:
        coder = DummyCoder(name="coder")
        decoder = DummyDecoder(name="decoder")

    # Модулятор/демодулятор
    mode_map = {
        'bpsk':  ('BPSK',  BPSKModulator,  BPSKDemodulator,  False),
        'dbpsk': ('DBPSK', BPSKModulator,  DBPSKDemodulator, True),
        'qpsk':  ('QPSK',  QPSKModulator,  QPSKDemodulator,  False),
        'dqpsk': ('DQPSK', DQPSKModulator, DQPSKDemodulator, True),
        'bfsk':  ('BFSK',  BFSKModulator,  BFSKDemodulator,  False),
        'dbfsk': ('DBFSK', BFSKModulator,  DBFSKDemodulator, True),
        'none':  ('NONE',  DummyModulator,  DummyDemodulator, False),
         'gfsk':  ('GFSK',  GFSKModulator,  BFSKDemodulator,  False),   # демодулятор как у BFSK
    }
    mode_name, mod_class, demod_class, is_diff = mode_map[mod_widget.value]

    # Общие параметры
    sps = sps_widget.value
    fs = fs_widget.value
    snr_db = snr_widget.value
    fc = fc_widget.value
    use_carrier = use_carrier_widget.value and mode_name != 'NONE'
    deviation = deviation_widget.value if mode_name in ('BFSK', 'DBFSK') else 25.0

    ctx = {"samples_per_symbol": sps, "fs": fs}
    if use_carrier:
        ctx["fc"] = fc

    blocks = [src]
    if isinstance(src, (TextSource, HexSource)):
        blocks.append(BytesToBits(name="bytes_to_bits"))

    blocks.append(coder)

    if is_diff and mode_name in ("DBPSK", "DBFSK"):
        blocks += [PrependBit(name="prepend", bit=0),
                   DifferentialEncoder(name="diff_enc", initial_state=0)]

    channel = channel_widget.value
    multipath_bpsk = (channel == 'multipath' and mode_name == 'BPSK')

    # ---- Модулятор и ZOH (только если НЕ многолучевой BPSK) ----
    if not multipath_bpsk:
        if mode_name in ('BPSK', 'DBPSK', 'NONE'):
            blocks.append(mod_class(name="mod"))
        elif mode_name == 'QPSK':
            blocks.append(QPSKModulator(name="mod"))
        elif mode_name == 'DQPSK':
            blocks.append(DQPSKModulator(name="mod"))
        elif mode_name == 'BFSK':
            blocks.append(BFSKModulator(name="mod", deviation_hz=deviation, samples_per_symbol=sps, fs=fs))
        elif mode_name == 'DBFSK':
            blocks.append(BFSKModulator(name="mod", deviation_hz=deviation, samples_per_symbol=sps, fs=fs))
        elif mode_name == 'GFSK':
            # Здесь можно запросить BT через дополнительный виджет, а пока используем значение по умолчанию
            bt = 0.5   # можно сделать виджет bt_widget
            blocks.append(GFSKModulator(name="mod", bt=bt, deviation_hz=deviation,
                                    samples_per_symbol=sps, fs=fs))
        if mode_name in ('BPSK', 'DBPSK', 'QPSK', 'DQPSK', 'NONE'):
            blocks.append(ZeroOrderHold(name="zoh", samples_per_symbol=sps))

        if use_carrier:
            blocks.append(IQModulator(name="iq_mod", fc=fc))

    # ---- Канал ----
    if channel == 'none':
        blocks.append(NoChannel(name="no_channel"))
    elif channel == 'awgn':
        blocks.append(AWGNChannel(name="awgn", snr_db=snr_db, seed=np.random.randint(0,1000)))
    elif channel == 'rayleigh':
        blocks += [RayleighFlatFadingChannel(name="fading", seed=7, block_size=block_size_widget.value),
                   AWGNChannel(name="awgn", snr_db=snr_db, seed=np.random.randint(0,1000))]
    elif channel == 'multipath':
        if mode_name != 'BPSK':
            print("Многолучевой канал с эквалайзером только для BPSK. Переключено на AWGN.")
            blocks.append(AWGNChannel(name="awgn", snr_db=snr_db, seed=42))
        else:
            # ---- Специальная сборка многолучевого BPSK ----
            if use_carrier:
                print("Предупреждение: многолучевой режим с несущей не поддерживается, несущая отключена.")
                use_carrier = False
            # Парсим лучи
            taps_str = multipath_widget.value
            try:
                taps = []
                for pair in taps_str.split(';'):
                    parts = pair.strip().split(',')
                    d = int(parts[0])
                    c = float(parts[1].strip())
                    taps.append((d, c))
            except:
                taps = [(0, 1.0), (40, 0.6)]
            preamble = np.array([1,0,1,0,1,1,0,0,1,1,0,1,0,1,1,0,
                                 1,1,0,0,1,0,1,1,0,1,1,0,1,0,0,1], dtype=np.uint8)
            blocks += [
                PrependArrayBlock(prepend=preamble, name="prepend_preamble"),
                BPSKModulator(name="mod"),
                ZeroOrderHold(name="zoh", samples_per_symbol=sps),
                MultipathChannel(taps=taps, name="multipath"),
                AWGNChannel(name="awgn", snr_db=snr_db, seed=42),
                IntegrateAndDump(name="int_dump", samples_per_symbol=sps),
                LMSLinearEqualizer(name="eq", num_taps=11, step=0.1, preamble=None, dd=True),
                RemovePreambleBlock(n=len(preamble), name="remove_preamble"),
                BPSKDemodulator(name="demod"),
            ]
 # Допплеровский сдвиг и ручная компенсация
    freq_offset = freq_offset_widget.value
    if channel != 'none' and freq_offset != 0:
        blocks.append(DopplerChannel(name="doppler", freq_offset_hz=freq_offset))
        if compensate_widget.value:
            blocks.append(FrequencyCorrector(name="freq_corr", freq_offset_hz=freq_offset))
    # ---- IQ демодулятор (если включён и не многолучевой BPSK) ----
    if use_carrier and not multipath_bpsk:
        blocks.append(IQDemodulator(name="iq_demod", fc=fc))

    # ---- Демодулятор / обработка (если ещё не добавлен) ----
    if not multipath_bpsk:
        if mode_name in ('BPSK', 'DBPSK', 'QPSK', 'DQPSK', 'NONE'):
            blocks.append(IntegrateAndDump(name="int_dump", samples_per_symbol=sps))
            if mode_name == 'BPSK':
                blocks.append(BPSKDemodulator(name="demod"))
            elif mode_name == 'DBPSK':
                blocks.append(DBPSKDemodulator(name="demod"))
            elif mode_name == 'QPSK':
                blocks.append(QPSKDemodulator(name="demod"))
            elif mode_name == 'DQPSK':
                blocks.append(DQPSKDemodulator(name="demod"))
            elif mode_name == 'NONE':
                blocks.append(DummyDemodulator(name="demod"))
        elif mode_name in ('BFSK', 'DBFSK'):
            if mode_name == 'BFSK':
                blocks.append(BFSKDemodulator(name="demod", deviation_hz=deviation, samples_per_symbol=sps, fs=fs))
            else:
                blocks.append(DBFSKDemodulator(name="demod", deviation_hz=deviation, samples_per_symbol=sps, fs=fs))

    # ---- Декодер ----
    blocks.append(decoder)

    # ---- Восстановление сообщения ----
    if not isinstance(src, RawBitSource):
        blocks += [BitsToBytes(name="bits_to_bytes"),
                   MessageSink(name="msg_sink", errors="replace")]

    return blocks, ctx

In [ ]:
def visualize_signal(signal, fs=1.0, mode="amplitude", title=""):
    signal = np.asarray(signal)

    n = len(signal)

    if n == 0:
        print("Пустой сигнал")
        return

    t = np.arange(n) / fs

    spectrum = np.fft.fftshift(np.fft.fft(signal))
    freqs = np.fft.fftshift(np.fft.fftfreq(n, d=1/fs))

    fig, ax = plt.subplots(1, 2, figsize=(14, 4))

    # ====================================
    # СЛЕВА — СПЕКТР
    # ====================================

    if mode == "amplitude":
        ax[0].plot(freqs, np.abs(spectrum))
        ax[0].set_title("Amplitude Spectrum")

    elif mode == "phase":
        ax[0].plot(freqs, np.unwrap(np.angle(spectrum)))
        ax[0].set_title("Phase Spectrum")

    ax[0].set_xlabel("Frequency [Hz]")
    ax[0].grid(True)

    # ====================================
    # СПРАВА — ВРЕМЕННАЯ ОБЛАСТЬ
    # ====================================

    if np.iscomplexobj(signal):
        ax[1].plot(t, np.real(signal), label="I")
        ax[1].plot(t, np.imag(signal), label="Q")
        ax[1].legend()
    else:
        ax[1].plot(t, signal)

    ax[1].set_title("Time Domain")
    ax[1].set_xlabel("Time [s]")
    ax[1].grid(True)

    fig.suptitle(title)

    plt.tight_layout()
    plt.show()

In [ ]:
# Ячейка: функция запуска и привязка кнопки (выполняйте после создания всех виджетов и out)

def run_simulation(b):
    out.clear_output(wait=True)
    with out:
        blocks, ctx = build_blocks_from_widgets()
        # Если есть эквалайзер, добавим преамбулу
        for blk in blocks:
            if isinstance(blk, LMSLinearEqualizer):
                preamble_bits = np.array([1,0,1,0,1,1,0,0,1,1,0,1,0,1,1,0,
                                          1,1,0,0,1,0,1,1,0,1,1,0,1,0,0,1], dtype=np.uint8)
                mod = BPSKModulator()
                preamble_syms = mod(preamble_bits)
                ctx["preamble"] = preamble_syms
                break

        capture = {}
        out_val = run(blocks, x=None, ctx=ctx, capture=capture)
        
        src = blocks[0]
        print('=== РЕЗУЛЬТАТЫ ===')
        if hasattr(src, 'message'):
            print(f"Передано: {src.message}")
        elif hasattr(src, 'bitstring'):
            print(f"Передано бит: {src.bitstring}")
        elif hasattr(src, 'hexstring'):
            print(f"Передано hex: {src.hexstring}")

        try:
            tx_bits = capture.get("bytes_to_bits", None)
            if tx_bits is None and hasattr(src, 'bitstring'):
                tx_bits = np.array([int(b) for b in src.bitstring], dtype=np.uint8)
        except:
            tx_bits = None

        decoded = None
        for blk in blocks:
            if isinstance(blk, (HammingDecoder, DummyDecoder)):
                decoded = capture.get(blk.name, None)
                break
        if tx_bits is not None and decoded is not None:
            m = min(len(tx_bits), len(decoded))
            errs = np.sum(tx_bits[:m] != decoded[:m])
            ber = errs/m if m else 0
            print(f"Битовые ошибки: {errs}/{m} (BER={ber:.2e})")
       
        if isinstance(src, TextSource) and isinstance(out_val, str):
            print(f"Принято: {out_val!r}")
        elif isinstance(src, RawBitSource) and decoded is not None:
            recovered_bits = decoded[:len(src.bitstring)]
            print(f"Принятые биты: {''.join(str(b) for b in recovered_bits)}")
        elif isinstance(src, HexSource) and isinstance(out_val, bytes):
            print(f"Принятые hex: {out_val.hex()}")
        # ====================================
        # ВИЗУАЛИЗАЦИЯ
        # ====================================

        selected_block = block_view_widget.value

        if selected_block not in capture:
            print(f"Блок '{selected_block}' не найден")
            print("Доступные блоки:", list(capture.keys()))
        else:
            visualize_signal(
                capture[selected_block],
                fs=ctx["fs"],
                mode=spectrum_widget.value,
                title=selected_block
            )
        # Графики
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6))
        if "zoh" in capture:
            ax1.plot(capture["zoh"][:200].real, label='TX (I)')
        elif "mod" in capture:
            ax1.plot(capture["mod"][:200].real, label='TX (I)', alpha=0.7)
            ax1.plot(capture["mod"][:200].imag, label='TX (Q)', alpha=0.5)
        ax1.set_title("Переданный сигнал")
        ax1.grid(True); ax1.legend()

        rx_sig = None
        for key in ("awgn", "multipath", "no_channel"):
            if key in capture:
                rx_sig = capture[key]
                break
        # Если ничего не нашли, но есть другие блоки – можно взять выход последнего блока
        if rx_sig is None:
            rx_sig = out   # out – результат run (выход последнего блока), может быть что угодно, но для графика сойдёт

        if rx_sig is not None:
            ax2.plot(rx_sig[:200].real, label='RX (I)', color='r')
            if np.iscomplexobj(rx_sig) and np.any(np.abs(rx_sig.imag) > 1e-12):
                ax2.plot(rx_sig[:200].imag, label='RX (Q)', color='m', alpha=0.5)
            ax2.set_title("Принятый сигнал")
            ax2.grid(True); ax2.legend()

        plt.tight_layout()
        plt.show()

run_button.on_click(run_simulation)